# 🦾 GR00T Policy — NVIDIA Foundation Models for Robots

NVIDIA GR00T is a family of vision-language-action (VLA) models that turn
camera images + a text instruction into robot motor commands. `strands-robots`
supports GR00T N1.5 and N1.6 in two modes:

| Mode | How It Works | When to Use |
|------|-------------|-------------|
| **Service** | ZMQ client → Docker container with Isaac-GR00T | No local GPU needed |
| **Local** | Load model directly on your GPU | Lowest latency, needs `isaac-gr00t` |

This notebook covers the configuration system and client — no GPU required to follow along.

## The Key Challenge: Sensor-to-Model Mapping

Every robot has different camera names and joint names. GR00T has its own
internal names. The data configuration system bridges this gap.

For example, your robot might call its cameras `"front_cam"` and `"wrist_cam"`,
but the model expects `"video.front"` and `"video.wrist"`. The data config defines
this mapping.

In [ ]:
from strands_robots.policies.groot.data_config import (
    DATA_CONFIG_MAP, load_data_config, create_custom_data_config
)

# 25+ robot embodiment configs ship with the library
print(f"{len(DATA_CONFIG_MAP)} pre-built configs:\n")
for name in sorted(DATA_CONFIG_MAP):
    cfg = DATA_CONFIG_MAP[name]
    n_cams = len(cfg.video_keys)
    n_state = len(cfg.state_keys)
    print(f"  {name:30s}  {n_cams} camera(s), {n_state} state group(s)")

In [ ]:
# Let's look at the SO-100 dual-camera config in detail
cfg = load_data_config("so100_dualcam")

print(f"Config: {cfg.name}")
print(f"  Cameras:      {cfg.video_keys}")
print(f"  State groups: {cfg.state_keys}")
print(f"  Action groups:{cfg.action_keys}")
print(f"  Action horizon: {len(cfg.action_indices)} steps")
print()

# This is what Isaac-GR00T actually receives:
mc = cfg.modality_config()
for modality, config in mc.items():
    print(f"  {modality:10s} → keys={config.modality_keys}")

## Custom Configs for New Robots

Building a robot that's not in the pre-built list? Create a custom config:

In [ ]:
custom = create_custom_data_config(
    name="my_3cam_arm",
    video_keys=["video.left", "video.right", "video.overhead"],
    state_keys=["state.arm_joints", "state.gripper"],
    action_keys=["action.arm_joints", "action.gripper"],
)
print(f"✅ Registered: {custom.name}")

# It's immediately available everywhere:
assert load_data_config("my_3cam_arm") is custom

## Explicit Observation & Action Mappings

For non-standard robots, you tell GR00T exactly how your sensor names map
to model keys:

```python
policy = Gr00tPolicy(
    data_config="so100_dualcam",
    model_path="nvidia/GR00T-N1.6-3B",

    # Robot sensor → Model input
    observation_mapping={
        "front_cam":       "video.front",      # your camera → model's video key
        "wrist_cam":       "video.wrist",
        "joint_positions": "state.single_arm",  # your state → model's state key
        "grip_pos":        "state.gripper",
    },

    # Model output → Robot actuator
    action_mapping={
        "action.single_arm": "joint_positions",  # model's action → your actuator
        "action.gripper":    "grip_pos",
    },
)
```

If you don't provide explicit mappings, auto-inference runs: exact name match first,
then positional fallback (with a logged warning).

## ZMQ Serialization

The service client sends observations over ZMQ using msgpack. NumPy arrays and
custom config objects survive the round-trip:

In [ ]:
try:
    from strands_robots.policies.groot.client import MsgSerializer
    import numpy as np

    # NumPy arrays serialize cleanly
    data = {
        "image": np.random.randint(0, 255, (480, 640, 3), dtype=np.uint8),
        "state": np.array([0.1, -0.3, 0.5], dtype=np.float32),
        "instruction": "pick up the red block",
    }
    encoded = MsgSerializer.to_bytes(data)
    decoded = MsgSerializer.from_bytes(encoded)

    print(f"Encoded: {len(encoded):,} bytes")
    print(f"Image match:  {np.array_equal(data['image'], decoded['image'])}")
    print(f"State match:  {np.allclose(data['state'], decoded['state'])}")
    print(f"Text match:   {data['instruction'] == decoded['instruction']}")
except ImportError:
    print("⏭ Requires: pip install 'strands-robots[groot-service]'")
    print("  (needs msgpack + pyzmq for ZMQ serialization demo)")

In [ ]:
try:
    from strands_robots.policies.groot.client import MsgSerializer
    from strands_robots.policies.groot.data_config import ModalityConfig

    mc = ModalityConfig(delta_indices=[0], modality_keys=["front", "wrist"])
    roundtrip = MsgSerializer.from_bytes(MsgSerializer.to_bytes({"config": mc}))
    print(f"ModalityConfig round-trip: {roundtrip['config'].modality_keys}")
except ImportError:
    print("⏭ Requires: pip install 'strands-robots[groot-service]'")

## Connecting to a GR00T Service

```python
from strands_robots.policies.groot.client import Gr00tInferenceClient

# Basic connection
client = Gr00tInferenceClient(host="localhost", port=5555)
client.ping()  # True if server is running

# With authentication (warns about plaintext if host != localhost)
client = Gr00tInferenceClient(host="10.0.0.5", port=5555, api_token="my-token")
# Token also read from GROOT_API_TOKEN env var

# Get actions from the model
actions = client.get_action(observation_dict)
```

## Managing GR00T Docker Containers

The `gr00t_inference` tool manages Docker containers for you:

```python
from strands import Agent
from strands_robots import gr00t_inference

agent = Agent(tools=[gr00t_inference])

agent("Start a GR00T inference server for the SO-100 robot on port 5555")
# → pulls container, mounts model, starts ZMQ service

agent("What GR00T containers are running?")
# → lists active containers with port mappings

agent("Stop all GR00T services")
```